In [5]:
from ollama import chat

stream = chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Why is the sky blue?'}],
    stream=True,
)

for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with the tiny molecules of gases in the Earth's atmosphere.

Here's what happens:

1. When sunlight enters the Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).
2. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths.
3. This is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.
4. As a result of this scattering, the blue light is dispersed throughout the atmosphere, giving the sky its blue color.

However, during sunrise and sunset, when the sun is lower on the horizon, the light has to travel through more of the Earth's atmosphere to reach our eyes. This means that even more blue light is scattered away, leaving mainly longer wavelengths (like red and orange) to reach our eyes, which i

KeyboardInterrupt: 

# Assistant
# Anaconda Assistant Response

## Approach to Evaluate LLMs for Phishing/Extortion Detection Using Ollama

I'll outline a structured approach to evaluate locally installed LLMs on their ability to identify phishing and extortion messages using Ollama.


In [2]:
import pandas as pd
import numpy as np
import ollama
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# Step 1: Create a dataset of example messages
phishing_extortion_examples = [
    {"message": "URGENT: Your account has been compromised. Click here to reset your password immediately: http://bit.ly/2xYz", "type": "phishing"},
    {"message": "We have access to your webcam and all your personal files. Send $500 in Bitcoin within 24 hours or we'll release everything.", "type": "extortion"},
    {"message": "This is your bank. We've noticed suspicious activity. Please verify your identity by providing your full SSN and account details.", "type": "phishing"},
    {"message": "All your files have been encrypted. Send $1000 to this wallet or lose everything forever. This is not a joke.", "type": "extortion"},
    {"message": "Congratulations! You've won a free iPhone. Click here to claim your prize now: www.free-prizes-claim.com", "type": "phishing"},
    {"message": "We know your secret. Pay $300 or we'll tell everyone what you did last summer. You have 48 hours.", "type": "extortion"},
    {"message": "Your package delivery failed. Click here to reschedule: tracking-delivery-service.net/reschedule", "type": "phishing"},
    {"message": "We've hacked your email and installed malware on your computer. Send $200 in gift cards or we'll corrupt all your files.", "type": "extortion"},
    {"message": "Your account will be suspended. Verify now: secure-account-verify.com/login", "type": "phishing"},
    {"message": "We've been watching you. Pay $400 in Bitcoin or the video goes to all your contacts. You have 24 hours to decide.", "type": "extortion"}
]

# Convert to DataFrame for easier handling
df = pd.DataFrame(phishing_extortion_examples)

# Step 2: List available models in Ollama
available_models = ollama.list()
print("Available models:", [model['name'] for model in available_models['models']])

# Step 3: Define evaluation function
def evaluate_model(model_name, messages, system_prompt=None):
    results = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_name}"):
        message = row['message']
        true_type = row['type']
        
        # Construct prompt
        prompt = f"""
        Analyze the following message and determine if it is a phishing attempt, extortion attempt, or neither.
        Respond with only one word: "phishing", "extortion", or "safe".
        
        Message: {message}
        """
        
        # Query the model
        try:
            response = ollama.chat(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt} if system_prompt else None,
                    {"role": "user", "content": prompt}
                ]
            )
            
            prediction = response['message']['content'].strip().lower()
            
            # Clean up prediction (extract just the classification word)
            if "phishing" in prediction:
                clean_prediction = "phishing"
            elif "extortion" in prediction:
                clean_prediction = "extortion"
            else:
                clean_prediction = "safe"
                
            correct = clean_prediction == true_type
            
            results.append({
                "message": message,
                "true_type": true_type,
                "prediction": clean_prediction,
                "correct": correct,
                "full_response": response['message']['content']
            })
            
        except Exception as e:
            print(f"Error with model {model_name}: {e}")
            results.append({
                "message": message,
                "true_type": true_type,
                "prediction": "error",
                "correct": False,
                "full_response": str(e)
            })
    
    return pd.DataFrame(results)

# Step 4: Evaluate models
models_to_evaluate = ["llama2", "mistral"]  # Replace with your actual model names
system_prompt = "You are a cybersecurity expert who identifies phishing and extortion attempts."

evaluation_results = {}
for model in models_to_evaluate:
    try:
        results = evaluate_model(model, df, system_prompt)
        evaluation_results[model] = results
        
        # Calculate accuracy
        accuracy = results['correct'].mean()
        print(f"{model} accuracy: {accuracy:.2f}")
        
        # Print classification report
        print(f"\nClassification Report for {model}:")
        print(classification_report(results['true_type'], results['prediction']))
        
    except Exception as e:
        print(f"Could not evaluate {model}: {e}")

# Step 5: Visualize results
plt.figure(figsize=(12, 6))
accuracies = [evaluation_results[model]['correct'].mean() for model in evaluation_results]
plt.bar(evaluation_results.keys(), accuracies)
plt.title('Model Accuracy in Identifying Phishing and Extortion Messages')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
for i, acc in enumerate(accuracies):
    plt.text(i, acc + 0.02, f'{acc:.2f}', ha='center')
plt.tight_layout()
plt.show()

# Step 6: Save results
with open('llm_phishing_evaluation_results.json', 'w') as f:
    json.dump({model: results.to_dict('records') for model, results in evaluation_results.items()}, f, indent=2)

KeyError: 'name'


### Key Components of this Approach:

1. **Dataset Creation**: I've created 10 example messages (5 phishing, 5 extortion) that represent common tactics.

2. **Model Evaluation**: The code evaluates each locally installed LLM using Ollama's API.

3. **Standardized Prompting**: Each model receives the same prompt structure to ensure fair comparison.

4. **Result Analysis**: The approach calculates accuracy and generates classification reports for each model.

5. **Visualization**: Results are visualized to easily compare model performance.

### Suggested Extensions:

- Add more diverse examples or use a public dataset of phishing/extortion messages
- Implement more metrics like false positive/negative rates
- Test different system prompts to see how they affect detection accuracy
- Add a confidence score analysis if your models provide confidence values
- Compare with a baseline rule-based approach

This approach allows you to systematically evaluate how well different LLMs can identify malicious messages, which is valuable for security applications.